# Out-of-distribution evaluation: 144-drug external screen

Score the 144-drug × 39-strain external screen (derived from Maier et al. 2018) using a FARM checkpoint trained on the full 10-cohort dataset.

Fill in the placeholders, then run cells top-to-bottom.

In [ ]:
# === Fill these in ===
CHECKPOINT_PATH = 'path/to/your/farm_checkpoint.h5'   # trained on the full 10-cohort set
DATA_ROOT       = 'path/to/FARM_dataset_v1'

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from farm import predict
from farm.eval import score

ext_dir = os.path.join(DATA_ROOT, 'external_screen')
pheno  = pd.read_csv(os.path.join(ext_dir, 'external_screen_144drugs_39strains_phenotype.tsv'), sep='\t')
feats  = np.load(os.path.join(ext_dir, 'external_screen_39strains_KOcluster_features.npz'))
strains_in_feats = list(feats['strains'])
presence = feats['presence']    # (39, 3287, 2) uint8
mutation = feats['mutation']    # (39, 3287, 2) uint8

# Reduce the (3287, 2) one-hot to a (1962,) gene-presence/mutation vector
# per the FARM input contract.  The exact mapping depends on the KO-cluster
# index file used when the model was trained.  Adjust as needed.
def to_1d(arr):
    return arr[..., 0].astype(np.uint8)

strain_to_feat = {
    s: (to_1d(presence[i])[:1962], to_1d(mutation[i])[:1962])
    for i, s in enumerate(strains_in_feats)
}
print(f'{len(strain_to_feat)} strains, {pheno.drug.nunique()} drugs, '
      f'{len(pheno)} (drug, strain) pairs')

In [ ]:
# Build the prediction inputs and run inference.
pairs = []
labels = []
keep_rows = []
for i, row in pheno.iterrows():
    if row['strain'] not in strain_to_feat:
        continue
    pres, mut = strain_to_feat[row['strain']]
    pairs.append((row['drug_smiles'], pres, mut))
    labels.append(int(row['label_binary']))
    keep_rows.append(i)
labels = np.array(labels)
print(f'Inference on {len(pairs)} pairs …')
preds = predict.predict_batch(pairs, checkpoint=CHECKPOINT_PATH, batch_size=128)
probs = np.array([p['probability_resistant'] for p in preds])

In [ ]:
# Overall metrics across all 144 × 39 pairs.
overall = score(labels, probs)
for k, v in overall.items():
    print(f'  {k:20s} {v:.4f}')

In [ ]:
# Per-drug breakdown.
result_df = pheno.iloc[keep_rows].copy()
result_df['predicted_prob']  = probs
per_drug = []
for drug, g in result_df.groupby('drug'):
    s = score(g.label_binary.values, g.predicted_prob.values)
    per_drug.append({'drug': drug, **s, 'n': len(g)})
per_drug = pd.DataFrame(per_drug).sort_values('accuracy', ascending=False)
per_drug.head(20)

In [ ]:
# Bar chart of accuracy per drug — analogous to Figure 6C.
plt.figure(figsize=(10, 24))
y = np.arange(len(per_drug))
plt.barh(y, per_drug['accuracy'])
plt.yticks(y, per_drug['drug'], fontsize=7)
plt.gca().invert_yaxis()
plt.xlabel('Accuracy')
plt.title('Per-drug accuracy on the 144-drug external screen')
plt.tight_layout(); plt.show()